# Bericht M2

In [ ]:
import polars as pl
import glob
import numpy as np
import pandas as pd

import plotly.express as px
import matplotlib.pyplot as plt
import warnings

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
files = sorted(glob.glob('./data/EV__*.parquet'))

lf = pl.scan_parquet(files)
schema = lf.collect_schema()

## Daten auswählen und vorbereiten
Standardisierung und Entfernen konstanter Features:

1. Als Features werden alle numerischen Spalten außer `Time` und dem Signal „Schweißen EIN" (`EIN`) verwendet.
2. Mittelwert und Standardabweichung werden pro Feature berechnet und in `summary` gespeichert.
3. Konstante Features werden aus der Liste `features` entfernt, danach wird `df_features` mit den verbleibenden Features neu erstellt.


In [ ]:
# Für jedes Feature den Mittelwert and die Standartabweichung berechnen
means = df_features.select([pl.col(col).mean().alias(col) for col in features]).collect(engine="streaming")
stds = df_features.select([pl.col(col).std().alias(col) for col in features]).collect(engine="streaming")

# polars-dataFrame --> numpy-array
means = np.array([means[col][0] for col in features])
stds = np.array([stds[col][0] for col in features])

means_np = np.asarray(means, dtype=np.float64)
stds_np = np.asarray(stds, dtype=np.float64)

summary = pl.DataFrame({
    "feature": features,
    "mean": means_np,
    "std": stds_np
})

# Features finden, deren Standardabweichung 0
# solche Features sind konstant und enthalten keine Variationen
const_features = (
    summary
    .filter(pl.col("std") == 0)
    .get_column("feature")
    .to_list()
    
)

# kosntante Features entfernen
features = [col for col in features if col not in const_features]

# Features erneut auswählen
# dabei werden nur nicht-konstante Features verwendet
df_features = lf.select("Time", EIN, *features).drop_nulls(subset=["Time", EIN])
df_features = df_features.with_columns(seq_id=(pl.col(EIN) & ~pl.col(EIN).shift(fill_value=False)).cum_sum() - 1)
df_features = df_features.filter(pl.col(EIN)).select("Time", "seq_id", *features)

## Sequenzen in eine gemeinsame Repräsentation überführen
Da Schweißsequenzen unterschiedlich lang sind lassen sie sich nicht direkt vergleichen. Deshalb wird jede Sequenz linear auf N = 300 gleichmäßige Zeitpunkte interpoliert. Die Interpolierten Features werden in eine Matrix `df_X` überführt; NAN-Werte werden dabei ausgelassen.

In [1]:
df_seq = (
    df_features
    .group_by("seq_id")
    .agg(start=pl.col("Time").min(), end=pl.col("Time").max())
    .sort("seq_id")
    .collect(engine="streaming")
)

def interpolate(N):
    """Lineare Interpolation jeder Schweißsequenz auf N Zeitpunkte"""

    # Matrix für alle Sequenzen und interpolierten Features erstellen
    # Zeile entspricht einer Schweißsequenz
    # dabei IDs der Schweißsequenzen speichern
    X = np.empty((len(df_seq), len(features) * N), dtype=np.float64)
    seq_ids = df_seq["seq_id"].to_list()

    # Features standardisieren
    norm = [((pl.col(col) - mean) / std).cast(pl.Float64).alias(col) for col, mean, std in zip(features, means_np, stds_np)]

    empty = []

    for i, row in enumerate(df_seq.iter_rows(named=True)):
        seq = lf.filter(pl.col("Time").is_between(row["start"], row["end"]))
        seq = seq.filter(pl.col(EIN))
        seq = seq.select("Time", *features)
        seq = seq.sort("Time")
        seq = seq.with_columns(norm)
        seq = seq.collect(engine="streaming")

        t = (seq["Time"] - seq["Time"][0]).dt.total_milliseconds().to_numpy()
        t_lin = np.linspace(0, t[-1], N)
        

        for j, feature in enumerate(features):
            y = seq[feature].to_numpy()
            
            # Maske für NAN-Werte
            # Prüfen ob Feature Messwerte enthält
            dummy = ~np.isnan(y)
            if not dummy.any():
                empty.append((row["seq_id"], feature))
                X[i, j*N:(j+1)*N] = 0.0
                continue
            X[i, j*N:(j+1)*N] = np.interp(t_lin, t[dummy], y[dummy])
    return X, seq_ids, empty

def build_df_X(N):
    X, seq_ids, _ = interpolate(N)

    cols = [
        f"{feature}_t_{i}"
        for feature in features
        for i in range(N)
    ]

    return pd.DataFrame(
        X,
        index=pd.Index(seq_ids, name="seq_id"),
        columns=cols,
    )

NameError: name 'df_features' is not defined

In [ ]:
N_SAMPLES = 300

df_X = build_df_X(N_SAMPLES)
X = df_X.to_numpy()
df_X.iloc[:10, ::N_SAMPLES]

## Dimensionsreduktion 
Zur Dimensionsreduktion wird PCA verwendet. Zunächst werden die Sequenzen mit 2 Hauptkomponenten dargestellt. Anschließend wird mit der Cumulative Explained Variance Ratio (CEVR) bestimmt, wie viele Hauptkomponenten für die weitere Analyse nötig sind: Es werden so viele Komponenten behalten, dass mindestens 95 % der Gesamtvarianz erklärt werden.

In [ ]:
# PCA für 2 Hauptkomponenten
pca2 = PCA(n_components=2)

# # Projektion auf die Hauptkomponenten
df_pca = pd.DataFrame(pca2.fit_transform(X), columns=["PC1", "PC2"], index=df_X.index)

fig = px.scatter(df_pca, x="PC1", y="PC2", hover_name=df_pca.index, width=750, height=450)
fig.update_traces(marker=dict(color="darkorchid"))
fig.show()

In [ ]:
PCA_ = PCA()
PCA_.fit(X)

# Cumulative Explained Variance Ratio berechnen 
cum_var = PCA_.explained_variance_ratio_.cumsum()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, color="darkorchid")

plt.axhline(y=0.9, linestyle="--", label="90% var", color="black")
plt.axhline(y=0.95, linestyle="--", label="95% var", color="orange")
plt.axhline(y=0.99, linestyle="--", label="99% var", color="green")
plt.xlabel("N")
plt.ylabel("CEV")
plt.grid(True)
plt.legend()
plt.show()

## Clustering
Für das Clustering werden die ersten n=160 Hauptkomponenten verwendet, die zusammen mindestens 95 % der Gesamtvarianz erklären. Auf dieser reduzierten Datenbasis wird der K-Means-Algorithmus angewendet. Die geeignete Clusteranzahl (k) wird mithilfe der Elbow-Methode abgeschätzt. Zur Untersuchung und zum Vergleich der Clusterstruktur werden die Ergebnisse für $k=2,3,4$ dargestellt.

In [ ]:
var = PCA_.explained_variance_ratio_

# Anzahl der Hauptkomponenten bestimmen, die mindestens 95 % der Gesamtvarianz haben
total = 0
n = 0

for v in var:
    total += v
    n += 1

    if  total >= 0.95:
        break
        
X_PCA = PCA_.transform(X)
# Nur die ersten n Hauptkomponenten für die Clustering
X_reduced = X_PCA[:, :n]

# Elbow-Methode zur Bestimmung einer geeigneten Clusteranzahl
def elbow(n_init=10, k_max=15, data=X_reduced):
    ks = np.arange(1,k_max+1)
    inertias = [KMeans(n_clusters=k, n_init=n_init, random_state=0).fit(data).inertia_ for k in ks]
    
    return ks, inertias

ks, inertias = elbow(n_init=50)
ret = np.array(inertias) / inertias[0]

# Visualisierung der Elbow-Methode
fig = px.line(x=ks, y=ret, markers=True, labels={"x": "k", "y": "inertia / inertia(k=1)"}, width=800, height=450)
fig.update_traces(marker=dict(color="darkorchid"))
fig.show()

In [ ]:
containers = []

# Clustering
for k in [2, 3, 4]:
    tmp = df_pca[["PC1", "PC2"]].copy()
    # K-Means-Clustering mit k Clustern durchführen
    # n_init=50: 50 unterschiedlichen Initialisierungen ausführen
    tmp["cluster"] = KMeans(n_clusters=k, n_init=50, random_state=0).fit_predict(X_reduced).astype(str)
    tmp["k"] = k
    containers.append(tmp)

main_container = pd.concat(containers)

fig = px.scatter(main_container, x="PC1", y="PC2", color="cluster", facet_col="k", hover_name=main_container.index)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.update_layout(showlegend=False)
fig.show()

## Sequenzen untersuchen
Um zu untersuchen, ob sich die Unterschiede in der reduzierten Darstellung auch in den ursprünglichen Zeitreihen widerspiegeln, werden Sequenzen ausgewählt, die im reduzierten Raum nahe beieinanderliegen, weit voneinander entfernt sind oder als mögliche Ausreißer erscheinen.

Die Auswahl erfolgt anhand der ersten beiden Hauptkomponenten (PC1 und PC2). Als möglicher Ausreißer wird die Sequenz mit dem größten mittleren Abstand zu ihren vier nächsten Nachbarn ausgewählt.

In [ ]:
# Zwei Sequenzen finden, die im PCA-Raum am nächsten beieinander liegen
def find_nearest_points(df):
    P = df[["PC1", "PC2"]].to_numpy()
    # Paarweise euklidische Abstände zwischen allen Punkten berechnen
    dist = pairwise_distances(P)
    # Selbstabstände ausschließen
    np.fill_diagonal(dist, np.inf)
    # Indzes bestimmen
    i, j = np.unravel_index(dist.argmin(), dist.shape)

    return df.index[i], df.index[j]

# Zwei Sequenzen finden, die im PCA-Raum am weitesten voneinander entfernt liegen
def find_farthest_points(df):
    P = df[["PC1", "PC2"]].to_numpy()
    dist = pairwise_distances(P)
    np.fill_diagonal(dist, -np.inf)
    i, j = np.unravel_index(dist.argmax(), dist.shape)

    return df.index[i], df.index[j]

def find_outlier(df, k=5):
    P = df[["PC1", "PC2"]].to_numpy()

    # K-nearest Algorithm
    nn = NearestNeighbors(n_neighbors=k).fit(P)
    dist, idx = nn.kneighbors(P)
    
    # Die erste Spalte enthält den Punkt selbs
    # Deshalb nur die Abstände zu den Nachbarn berechnen
    mean_dist = dist[:, 1:].mean(axis=1)

    # Punkt mit dem größten mittleren Nachbarabstand auswählen
    i = mean_dist.argmax()
    # Den nächsten Nachbarn bestimmen
    j = idx[i, 1]

    return df.index[i], df.index[j]

In [ ]:
pairs = {
    "Nearest": find_nearest_points(df_pca),
    "Farthest": find_farthest_points(df_pca),
    "Outlier": find_outlier(df_pca),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)

for ax, (name, (i, j)) in zip(axes, pairs.items()):
    ax.scatter(df_pca["PC1"], df_pca["PC2"],color="lightgray", s=15)
    ax.scatter(df_pca.loc[i, "PC1"], df_pca.loc[i, "PC2"], color="darkorchid", s=40, label="seq_i",)
    ax.scatter(df_pca.loc[j, "PC1"], df_pca.loc[j, "PC2"], color="orange", s=40, label="seq_j",)

    ax.set_title(name)
    ax.set_xlabel("PC1")
    ax.legend()

axes[0].set_ylabel("PC2")

plt.tight_layout()
plt.show()

## Visualisierung der Daten
Hilfsfunktionen zum Laden und Plotten der ursprünglichen und interpolierten Sequenzen.

In [ ]:
def get_orig_seq(seq_id):
    row = df_seq.filter(pl.col("seq_id") == seq_id).row(0, named=True)

    seq = lf.filter(pl.col("Time").is_between(row["start"], row["end"]))
    seq = seq.filter(pl.col(EIN))
    seq = seq.select("Time", *features)
    seq = seq.sort("Time")
    seq = seq.collect(engine="streaming")

    return seq


def get_interp_seq(seq_id):
    return df_X.loc[seq_id]


def plot_orig_and_interp_seq(seq_i, seq_j, interp_i, interp_j, i, j, feature):
    cols = [f"{feature}_t_{k}" for k in range(N_SAMPLES)]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))

    for seq, interp, seq_id, color in [(seq_i, interp_i, i, "darkorchid"),(seq_j, interp_j, j, "orange"),]:
        t = (seq["Time"] - seq["Time"][0]).dt.total_milliseconds().to_numpy()
        ax1.plot(t, seq[feature].to_numpy(), color=color, label=f"seq_id={seq_id}") # orig
        ax2.plot(np.arange(N_SAMPLES), interp[cols].to_numpy(), color=color, label=f"seq_id={seq_id}") # interp

    ax1.set_title("Orig")
    ax1.set_ylabel(feature)
    ax1.legend()
    ax1.grid(alpha=0.6)

    ax2.set_title("Interp.")
    ax2.set_xlabel("t")
    ax2.legend()
    ax2.grid(alpha=0.6)

    plt.tight_layout()
    plt.show()

In [2]:
i, j = pairs["Nearest"]
plot_orig_and_interp_seq(get_orig_seq(i), get_orig_seq(j), get_interp_seq(i), get_interp_seq(j), i, j, features[2])

i, j = pairs["Farthest"]
plot_orig_and_interp_seq(get_orig_seq(i), get_orig_seq(j), get_interp_seq(i), get_interp_seq(j), i, j, features[2])

i, j = pairs["Outlier"]
plot_orig_and_interp_seq(get_orig_seq(i), get_orig_seq(j), get_interp_seq(i), get_interp_seq(j), i, j, features[2])


NameError: name 'pairs' is not defined

## Ergebnisse
Beim Vergleich der ursprünglichen Zeitreihen lassen sich einige Unterschiede aus der PCA-Darstellung wiedererkennen. Die beiden am nächsten beieinanderliegenden Sequenzen zeigen ähnliche Verläufe. Das am weitesten voneinander entfernte Paar unterscheidet sich dagegen deutlich in der Höhe und Dauer des Signals.

Beim möglichen Ausreißer lassen sich Unterschiede im betrachteten Merkmal kaum erkennen. Möglicherweise wird
seine Lage in der PCA-Darstellung stärker durch andere Merkmale beeinflusst. Um das genauer zu beurteilen, müssten auch diese betrachtet werden.